In [2]:
import pandas as pd
import numpy as np
import json
import catboost as cb
import os

# --- 1. ตั้งค่า Path และโหลด Model ---
MODEL_PATH = '../models/wildfire_improved_model_d.cbm'
POINTS_JSON_PATH = '../data/district_points_data.json'
BASELINE_PATH = '../data/baseline_table.csv' # สำหรับคำนวณค่าความผิดปกติ

model = cb.CatBoostClassifier()
model.load_model(MODEL_PATH)
baseline_df = pd.read_csv(BASELINE_PATH)

# --- 2. อ่านข้อมูลจุดพิกัดรายอำเภอ ---
with open(POINTS_JSON_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

all_points = []
for province, districts in raw_data.items():
    for district, points in districts.items():
        for pt in points:
            # ตรวจสอบและเติมค่าพื้นฐานถ้าไม่มีคีย์ใน JSON
            pt['NAME_1'] = province
            pt['NAME_2'] = district
            pt['slope'] = pt.get('slope', 5)   # ถ้าไม่มี slope ให้ใช้ค่า 5 (องศา) เป็นค่าเริ่มต้น
            pt['elev'] = pt.get('elev', 200)  # ถ้าไม่มี elev ให้ใช้ค่า 200 (เมตร) เป็นค่าเริ่มต้น
            pt['temp'] = pt.get('temp', 30)
            pt['ndvi'] = pt.get('ndvi', 0.4)
            all_points.append(pt)

df = pd.DataFrame(all_points)
print(f"Loaded {len(df)} points.")

# --- 3. เตรียม Features ---
# ตอนนี้ df จะมีคอลัมน์ slope และ elev แน่นอนแล้ว
df['terrain_roughness'] = df['slope'] * np.log1p(df['elev'])
# ... (โค้ดส่วนที่เหลือเหมือนเดิม) ...

df['wind_speed'] = 2.5
df['cluster_id'] = "1" # พื้นที่เสี่ยงสูง
df['landcover'] = "10" # ป่า

# จัดเรียงคอลัมน์ให้ตรงกับโมเดล (32 Features)
FEATURE_NAMES = [
    'ndvi', 'ndwi', 'nbr', 'blue', 'green', 'red', 'nir', 'swir1', 'swir2', 
    'temp', 'soil_moisture', 'wind_u', 'wind_v', 'elev', 'slope', 'aspect', 
    'landcover', 'month', 'NAME_1', 'NAME_2', 'veg_stress', 'fire_weather_idx', 
    'wind_speed', 'drought_proxy', 'terrain_roughness', 'hot_dry_stress', 
    'ndvi_anomaly', 'moisture_anomaly', 'temp_anomaly', 'month_sin', 'month_cos', 'cluster_id'
]

# เติมค่าที่ขาด (Fill Missing)
for col in FEATURE_NAMES:
    if col not in df.columns:
        df[col] = 0

X = df[FEATURE_NAMES]

# --- 4. ทำนายความเสี่ยงด้วยโมเดลจริง ---
print("Predicting risk probabilities...")
df['predicted_risk'] = model.predict_proba(X)[:, 1]

# --- 5. สรุปผลรายอำเภอ (Aggregation) ---
# ปรับสเกลความเสี่ยงให้เป็น 0-100 (ตามที่เราคุยกันไว้เพื่อให้แสดงผลสวยงาม)
df['display_risk'] = (df['predicted_risk'] / 0.22) * 100
df.loc[df['display_risk'] > 95, 'display_risk'] = 95

district_summary = df.groupby(['NAME_1', 'NAME_2'])['display_risk'].mean().reset_index()

# แปลงเป็น JSON Format: { "Province": { "District": risk_value } }
final_json = {}
for _, row in district_summary.iterrows():
    p, d, r = row['NAME_1'], row['NAME_2'], row['display_risk']
    if p not in final_json:
        final_json[p] = {}
    final_json[p][d] = round(float(r), 2)

# --- 6. บันทึกไฟล์ ---
with open('../data/district_risk_summary.json', 'w', encoding='utf-8') as f:
    json.dump(final_json, f, ensure_ascii=False, indent=2)

print("✅ Success! '../data/district_risk_summary.json' has been created.")


Loaded 3941 points.
Predicting risk probabilities...
✅ Success! '../data/district_risk_summary.json' has been created.
